In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.ticker import PercentFormatter

import statsmodels.api as sm

from statsmodels.stats.multitest import (
    multipletests
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

FINAL_FIGURE_DIR = (
    PROJECT_ROOT
    / "reports"
    / "figures"
    / "final_comparison"
)

FINAL_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Final-model comparison environment was initialized.")

In [ ]:
EVALUATION_PANEL_FILE = (
    PROCESSED_DATA_DIR
    / "19_strategy_benchmark_evaluation_panel_2015_2025.parquet"
)

ROBUSTNESS_MONTHLY_FILE = (
    PROCESSED_DATA_DIR
    / "34_robustness_monthly_returns.csv"
)

ML_MONTHLY_FILE = (
    PROCESSED_DATA_DIR
    / "40_ml_portfolio_monthly_returns.csv"
)

FF_FACTOR_FILE = (
    PROCESSED_DATA_DIR
    / "15_fama_french_monthly_2015_2025.parquet"
)

evaluation_panel_df = pd.read_parquet(
    EVALUATION_PANEL_FILE
)

robustness_monthly_df = pd.read_csv(
    ROBUSTNESS_MONTHLY_FILE
)

ml_monthly_df = pd.read_csv(
    ML_MONTHLY_FILE
)

ff_factor_df = pd.read_parquet(
    FF_FACTOR_FILE
)

evaluation_panel_df["return_month"] = pd.to_datetime(
    evaluation_panel_df["return_month"]
)

robustness_monthly_df["return_month"] = pd.to_datetime(
    robustness_monthly_df["return_month"]
)

ml_monthly_df["return_month"] = pd.to_datetime(
    ml_monthly_df["return_month"]
)

ff_factor_df["month"] = pd.to_datetime(
    ff_factor_df["month"]
)

print("Final comparison datasets were loaded successfully.")
print("Evaluation-panel rows:", len(evaluation_panel_df))
print("Robustness rows:", len(robustness_monthly_df))
print("Machine-learning rows:", len(ml_monthly_df))
print("Fama-French rows:", len(ff_factor_df))

In [ ]:
final_series_frames = []

benchmark_series_specifications = {
    "CRSP S&P 500 Value Weighted":
        "crsp_value_weighted_total_return",

    "CRSP S&P 500 Equal Weighted":
        "crsp_equal_weighted_total_return"
}

for series_name, return_column in (
    benchmark_series_specifications.items()
):
    benchmark_frame = pd.DataFrame({
        "return_month":
            evaluation_panel_df[
                "return_month"
            ],

        "series":
            series_name,

        "net_return":
            evaluation_panel_df[
                return_column
            ],

        "turnover":
            np.nan,

        "model_category":
            "Official Benchmark",

        "selection_status":
            "Benchmark"
    })

    final_series_frames.append(
        benchmark_frame
    )


ml_strategy_name_mapping = {
    "Quality Factor Top 20%":
        (
            "Quality Baseline "
            "(Top 20% Equal Weight)"
        ),

    "Traditional Six-Factor Top 20%":
        (
            "Six-Factor Baseline "
            "(Top 20% Equal Weight)"
        ),

    "Ridge Prediction Top 20%":
        (
            "Ridge ML "
            "(Top 20% Equal Weight)"
        ),

    "XGBoost Prediction Top 20%":
        (
            "XGBoost ML "
            "(Top 20% Equal Weight)"
        )
}

for original_name, final_name in (
    ml_strategy_name_mapping.items()
):
    strategy_sample = (
        ml_monthly_df
        .loc[
            ml_monthly_df[
                "strategy"
            ]
            == original_name
        ]
        .sort_values("return_month")
        .copy()
    )

    if len(strategy_sample) != 132:
        raise ValueError(
            f"{original_name} should contain 132 months."
        )

    if "ML" in final_name:
        model_category = "Machine Learning"
    else:
        model_category = "Traditional Factor"

    strategy_frame = pd.DataFrame({
        "return_month":
            strategy_sample[
                "return_month"
            ],

        "series":
            final_name,

        "net_return":
            strategy_sample[
                "net_return"
            ],

        "turnover":
            strategy_sample[
                "turnover"
            ],

        "model_category":
            model_category,

        "selection_status":
            "Pre-Specified"
    })

    final_series_frames.append(
        strategy_frame
    )


robustness_winner_name = (
    "Quality | Top 10% | Inverse Volatility"
)

robustness_winner_sample = (
    robustness_monthly_df
    .loc[
        (
            robustness_monthly_df[
                "strategy"
            ]
            == robustness_winner_name
        )
        & (
            robustness_monthly_df[
                "cost_label"
            ]
            == "10 bps"
        )
    ]
    .sort_values("return_month")
    .copy()
)

if len(robustness_winner_sample) != 132:
    raise ValueError(
        "The robustness winner should contain 132 months."
    )

robustness_winner_frame = pd.DataFrame({
    "return_month":
        robustness_winner_sample[
            "return_month"
        ],

    "series":
        (
            "Quality Robustness Winner "
            "(Top 10% Inverse Volatility)"
        ),

    "net_return":
        robustness_winner_sample[
            "net_return"
        ],

    "turnover":
        robustness_winner_sample[
            "turnover"
        ],

    "model_category":
        "Traditional Factor",

    "selection_status":
        "Ex-Post Sensitivity Winner"
})

final_series_frames.append(
    robustness_winner_frame
)

final_return_long_df = pd.concat(
    final_series_frames,
    ignore_index=True
)

final_return_long_df = (
    final_return_long_df
    .sort_values(
        [
            "series",
            "return_month"
        ]
    )
    .reset_index(drop=True)
)

print("Unified final return dataset was created.")
print(
    "Number of return series:",
    final_return_long_df[
        "series"
    ].nunique()
)
print(
    "Number of monthly records:",
    len(final_return_long_df)
)

In [ ]:
duplicate_final_records = (
    final_return_long_df
    .duplicated(
        subset=[
            "series",
            "return_month"
        ]
    )
    .sum()
)

series_validation_df = (
    final_return_long_df
    .groupby("series")
    .agg(
        number_of_months=(
            "return_month",
            "nunique"
        ),

        start_month=(
            "return_month",
            "min"
        ),

        end_month=(
            "return_month",
            "max"
        ),

        missing_returns=(
            "net_return",
            lambda values: values.isna().sum()
        ),

        average_turnover=(
            "turnover",
            "mean"
        )
    )
)

if duplicate_final_records != 0:
    raise ValueError(
        "Duplicate final series-month records were found."
    )

if (
    series_validation_df[
        "number_of_months"
    ]
    != 132
).any():
    raise ValueError(
        "Every final series should contain 132 months."
    )

if (
    series_validation_df[
        "missing_returns"
    ]
    != 0
).any():
    raise ValueError(
        "Missing final portfolio returns were found."
    )

print("Final return-panel validation:")
print(
    "Duplicate series-month records:",
    duplicate_final_records
)

display(series_validation_df)

In [ ]:
final_factor_df = (
    ff_factor_df
    .rename(
        columns={
            "month":
                "return_month"
        }
    )
)

final_evaluation_long_df = (
    final_return_long_df
    .merge(
        final_factor_df,
        on="return_month",
        how="left",
        validate="many_to_one"
    )
)

equal_weighted_benchmark_df = (
    evaluation_panel_df[
        [
            "return_month",
            "crsp_equal_weighted_total_return"
        ]
    ]
    .copy()
)

final_evaluation_long_df = (
    final_evaluation_long_df
    .merge(
        equal_weighted_benchmark_df,
        on="return_month",
        how="left",
        validate="many_to_one"
    )
)

required_final_factor_columns = [
    "rf",
    "mkt_rf",
    "smb",
    "hml",
    "rmw",
    "cma",
    "mom"
]

missing_final_factor_values = (
    final_evaluation_long_df[
        required_final_factor_columns
    ]
    .isna()
    .sum()
)

if missing_final_factor_values.sum() != 0:
    raise ValueError(
        "Missing final factor observations were found."
    )

print("Risk-free rate and factor data were merged successfully.")
print("\nMissing factor values:")
print(missing_final_factor_values)

In [ ]:
def calculate_final_performance(
    group
):
    group = (
        group
        .sort_values("return_month")
        .copy()
    )

    returns = (
        group["net_return"]
        .astype(float)
    )

    risk_free_rate = (
        group["rf"]
        .astype(float)
    )

    excess_returns = (
        returns
        - risk_free_rate
    )

    benchmark_returns = (
        group[
            "crsp_equal_weighted_total_return"
        ]
        .astype(float)
    )

    active_returns = (
        returns
        - benchmark_returns
    )

    number_of_months = len(returns)

    terminal_wealth = (
        1.0 + returns
    ).prod()

    annualized_return = (
        terminal_wealth
        ** (
            12.0
            / number_of_months
        )
        - 1.0
    )

    annualized_volatility = (
        returns.std(ddof=1)
        * np.sqrt(12.0)
    )

    annualized_sharpe = (
        excess_returns.mean()
        / excess_returns.std(ddof=1)
        * np.sqrt(12.0)
    )

    downside_returns = np.minimum(
        excess_returns,
        0.0
    )

    downside_deviation = (
        np.sqrt(
            np.mean(
                downside_returns ** 2
            )
        )
        * np.sqrt(12.0)
    )

    annualized_sortino = (
        excess_returns.mean()
        * 12.0
        / downside_deviation
        if downside_deviation > 0
        else np.nan
    )

    wealth = (
        1.0 + returns
    ).cumprod()

    drawdown = (
        wealth
        / wealth.cummax()
        - 1.0
    )

    fifth_percentile = (
        returns.quantile(0.05)
    )

    historical_cvar_95 = (
        -returns[
            returns <= fifth_percentile
        ].mean()
    )

    tracking_error = (
        active_returns.std(ddof=1)
        * np.sqrt(12.0)
    )

    information_ratio = (
        active_returns.mean()
        * 12.0
        / tracking_error
        if tracking_error > 0
        else np.nan
    )

    return pd.Series({
        "number_of_months":
            number_of_months,

        "annualized_return":
            annualized_return,

        "annualized_volatility":
            annualized_volatility,

        "annualized_sharpe":
            annualized_sharpe,

        "annualized_sortino":
            annualized_sortino,

        "maximum_drawdown":
            drawdown.min(),

        "historical_cvar_95":
            historical_cvar_95,

        "average_turnover":
            group["turnover"].mean(),

        "annualized_active_return":
            active_returns.mean() * 12.0,

        "annualized_tracking_error":
            tracking_error,

        "information_ratio":
            information_ratio,

        "positive_month_rate":
            (returns > 0).mean(),

        "terminal_wealth":
            terminal_wealth
    })


final_performance_summary_df = (
    final_evaluation_long_df
    .groupby(
        [
            "series",
            "model_category",
            "selection_status"
        ],
        sort=True
    )
    .apply(
        calculate_final_performance
    )
    .reset_index()
    .sort_values(
        "annualized_sharpe",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Final unified performance summary:")
display(
    final_performance_summary_df.round(4)
)

In [ ]:
final_subperiods = {
    "2015-2019 Pre-COVID": (
        pd.Timestamp("2015-01-31"),
        pd.Timestamp("2019-12-31")
    ),

    "2020-2022 Stress Period": (
        pd.Timestamp("2020-01-31"),
        pd.Timestamp("2022-12-31")
    ),

    "2023-2025 Recent Period": (
        pd.Timestamp("2023-01-31"),
        pd.Timestamp("2025-12-31")
    )
}


def calculate_subperiod_performance(
    group
):
    group = (
        group
        .sort_values("return_month")
        .copy()
    )

    returns = (
        group["net_return"]
        .astype(float)
    )

    excess_returns = (
        returns
        - group["rf"].astype(float)
    )

    terminal_wealth = (
        1.0 + returns
    ).prod()

    annualized_return = (
        terminal_wealth
        ** (
            12.0
            / len(returns)
        )
        - 1.0
    )

    annualized_volatility = (
        returns.std(ddof=1)
        * np.sqrt(12.0)
    )

    annualized_sharpe = (
        excess_returns.mean()
        / excess_returns.std(ddof=1)
        * np.sqrt(12.0)
    )

    cumulative_wealth = (
        1.0 + returns
    ).cumprod()

    wealth_values = np.concatenate(
        [
            np.array([1.0]),
            cumulative_wealth.to_numpy(
                dtype=float
            )
        ]
    )

    running_peak_values = (
        np.maximum.accumulate(
            wealth_values
        )
    )

    drawdown_values = (
        wealth_values
        / running_peak_values
        - 1.0
    )

    return pd.Series({
        "number_of_months":
            len(returns),

        "annualized_return":
            annualized_return,

        "annualized_volatility":
            annualized_volatility,

        "annualized_sharpe":
            annualized_sharpe,

        "maximum_drawdown":
            drawdown_values.min()
    })


subperiod_records = []

for period_name, dates in (
    final_subperiods.items()
):
    start_date, end_date = dates

    period_sample = (
        final_evaluation_long_df
        .loc[
            (
                final_evaluation_long_df[
                    "return_month"
                ]
                >= start_date
            )
            & (
                final_evaluation_long_df[
                    "return_month"
                ]
                <= end_date
            )
        ]
        .copy()
    )

    period_summary = (
        period_sample
        .groupby("series")
        .apply(
            calculate_subperiod_performance
        )
        .reset_index()
    )

    period_summary[
        "subperiod"
    ] = period_name

    subperiod_records.append(
        period_summary
    )

final_subperiod_performance_df = pd.concat(
    subperiod_records,
    ignore_index=True
)

subperiod_return_pivot_df = (
    final_subperiod_performance_df
    .pivot(
        index="series",
        columns="subperiod",
        values="annualized_return"
    )
)

subperiod_sharpe_pivot_df = (
    final_subperiod_performance_df
    .pivot(
        index="series",
        columns="subperiod",
        values="annualized_sharpe"
    )
)

print("Subperiod annualized returns:")
display(
    subperiod_return_pivot_df.round(4)
)

print("Subperiod annualized Sharpe ratios:")
display(
    subperiod_sharpe_pivot_df.round(4)
)

In [ ]:
final_return_wide_df = (
    final_return_long_df
    .pivot(
        index="return_month",
        columns="series",
        values="net_return"
    )
    .sort_index()
)

equal_weighted_series_name = (
    "CRSP S&P 500 Equal Weighted"
)

candidate_strategy_names = [
    "Quality Baseline (Top 20% Equal Weight)",

    (
        "Quality Robustness Winner "
        "(Top 10% Inverse Volatility)"
    ),

    (
        "Six-Factor Baseline "
        "(Top 20% Equal Weight)"
    ),

    "Ridge ML (Top 20% Equal Weight)",

    "XGBoost ML (Top 20% Equal Weight)"
]

equal_weighted_returns = (
    final_return_wide_df[
        equal_weighted_series_name
    ]
)

active_return_test_records = []

for strategy_name in candidate_strategy_names:
    active_returns = (
        final_return_wide_df[
            strategy_name
        ]
        - equal_weighted_returns
    ).dropna()

    constant = np.ones(
        shape=(
            len(active_returns),
            1
        )
    )

    model = sm.OLS(
        active_returns.to_numpy(),
        constant
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 3
        }
    )

    tracking_error = (
        active_returns.std(ddof=1)
        * np.sqrt(12.0)
    )

    annualized_active_return = (
        active_returns.mean()
        * 12.0
    )

    information_ratio = (
        annualized_active_return
        / tracking_error
        if tracking_error > 0
        else np.nan
    )

    active_return_test_records.append({
        "series":
            strategy_name,

        "number_of_months":
            len(active_returns),

        "annualized_active_return":
            annualized_active_return,

        "newey_west_t_statistic":
            model.tvalues[0],

        "raw_p_value":
            model.pvalues[0],

        "annualized_tracking_error":
            tracking_error,

        "information_ratio":
            information_ratio,

        "positive_active_month_rate":
            (active_returns > 0).mean()
    })

active_return_test_df = pd.DataFrame(
    active_return_test_records
)

holm_rejections, holm_p_values, _, _ = (
    multipletests(
        active_return_test_df[
            "raw_p_value"
        ].to_numpy(),
        alpha=0.05,
        method="holm"
    )
)

active_return_test_df[
    "holm_adjusted_p_value"
] = holm_p_values

active_return_test_df[
    "significant_after_holm_at_5_percent"
] = holm_rejections

active_return_test_df = (
    active_return_test_df
    .sort_values(
        "annualized_active_return",
        ascending=False
    )
    .reset_index(drop=True)
)

print("HAC active-return tests with Holm correction:")
display(
    active_return_test_df.round(4)
)

In [ ]:
final_factor_columns = [
    "mkt_rf",
    "smb",
    "hml",
    "rmw",
    "cma",
    "mom"
]

final_alpha_records = []

for series_name, group in (
    final_evaluation_long_df
    .groupby(
        "series",
        sort=True
    )
):
    regression_sample = (
        group
        .dropna(
            subset=[
                "net_return",
                "rf"
            ]
            + final_factor_columns
        )
        .sort_values("return_month")
    )

    excess_return = (
        regression_sample["net_return"]
        - regression_sample["rf"]
    )

    independent_variables = sm.add_constant(
        regression_sample[
            final_factor_columns
        ],
        has_constant="add"
    )

    model = sm.OLS(
        excess_return,
        independent_variables
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 3
        }
    )

    final_alpha_records.append({
        "series":
            series_name,

        "annualized_alpha":
            model.params["const"] * 12.0,

        "alpha_t_statistic":
            model.tvalues["const"],

        "raw_alpha_p_value":
            model.pvalues["const"],

        "mkt_rf_beta":
            model.params["mkt_rf"],

        "smb_beta":
            model.params["smb"],

        "hml_beta":
            model.params["hml"],

        "rmw_beta":
            model.params["rmw"],

        "cma_beta":
            model.params["cma"],

        "mom_beta":
            model.params["mom"],

        "adjusted_r_squared":
            model.rsquared_adj
    })

final_alpha_df = pd.DataFrame(
    final_alpha_records
)

final_alpha_df[
    "holm_adjusted_alpha_p_value"
] = np.nan

final_alpha_df[
    "significant_alpha_after_holm_at_5_percent"
] = False

candidate_alpha_mask = (
    final_alpha_df[
        "series"
    ]
    .isin(candidate_strategy_names)
)

candidate_alpha_p_values = (
    final_alpha_df.loc[
        candidate_alpha_mask,
        "raw_alpha_p_value"
    ]
    .to_numpy()
)

alpha_rejections, alpha_adjusted_p_values, _, _ = (
    multipletests(
        candidate_alpha_p_values,
        alpha=0.05,
        method="holm"
    )
)

final_alpha_df.loc[
    candidate_alpha_mask,
    "holm_adjusted_alpha_p_value"
] = alpha_adjusted_p_values

final_alpha_df.loc[
    candidate_alpha_mask,
    "significant_alpha_after_holm_at_5_percent"
] = alpha_rejections

final_alpha_df = (
    final_alpha_df
    .sort_values(
        "annualized_alpha",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Final factor-adjusted alpha results:")
display(
    final_alpha_df.round(4)
)

In [ ]:
equal_weighted_performance = (
    final_performance_summary_df
    .loc[
        final_performance_summary_df[
            "series"
        ]
        == equal_weighted_series_name
    ]
    .iloc[0]
)

candidate_performance_df = (
    final_performance_summary_df
    .loc[
        final_performance_summary_df[
            "series"
        ]
        .isin(candidate_strategy_names)
    ]
    .copy()
)

active_test_for_merge_df = (
    active_return_test_df[
        [
            "series",
            "annualized_active_return",
            "raw_p_value",
            "holm_adjusted_p_value",
            "significant_after_holm_at_5_percent"
        ]
    ]
    .rename(
        columns={
            "annualized_active_return":
                "hac_annualized_active_return",

            "raw_p_value":
                "active_return_raw_p_value",

            "holm_adjusted_p_value":
                "active_return_holm_p_value",

            "significant_after_holm_at_5_percent":
                "significant_active_return_after_holm"
        }
    )
)

alpha_for_merge_df = (
    final_alpha_df[
        [
            "series",
            "annualized_alpha",
            "raw_alpha_p_value",
            "holm_adjusted_alpha_p_value",
            "significant_alpha_after_holm_at_5_percent"
        ]
    ]
    .copy()
)

final_evidence_df = (
    candidate_performance_df
    .merge(
        active_test_for_merge_df,
        on="series",
        how="left",
        validate="one_to_one"
    )
    .merge(
        alpha_for_merge_df,
        on="series",
        how="left",
        validate="one_to_one"
    )
)

final_evidence_df[
    "return_above_equal_weight"
] = (
    final_evidence_df[
        "annualized_return"
    ]
    > equal_weighted_performance[
        "annualized_return"
    ]
)

final_evidence_df[
    "sharpe_above_equal_weight"
] = (
    final_evidence_df[
        "annualized_sharpe"
    ]
    > equal_weighted_performance[
        "annualized_sharpe"
    ]
)

final_evidence_df[
    "drawdown_better_than_equal_weight"
] = (
    final_evidence_df[
        "maximum_drawdown"
    ]
    > equal_weighted_performance[
        "maximum_drawdown"
    ]
)

final_evidence_df[
    "positive_active_return"
] = (
    final_evidence_df[
        "hac_annualized_active_return"
    ]
    > 0
)

final_evidence_df[
    "statistically_significant_active_return"
] = (
    final_evidence_df[
        "significant_active_return_after_holm"
    ]
)

final_evidence_df[
    "positive_factor_alpha"
] = (
    final_evidence_df[
        "annualized_alpha"
    ]
    > 0
)

final_evidence_df[
    "statistically_significant_positive_alpha"
] = (
    (
        final_evidence_df[
            "annualized_alpha"
        ]
        > 0
    )
    & (
        final_evidence_df[
            "significant_alpha_after_holm_at_5_percent"
        ]
    )
)

evidence_display_columns = [
    "series",
    "selection_status",
    "annualized_return",
    "annualized_sharpe",
    "maximum_drawdown",
    "hac_annualized_active_return",
    "active_return_holm_p_value",
    "annualized_alpha",
    "holm_adjusted_alpha_p_value",
    "return_above_equal_weight",
    "sharpe_above_equal_weight",
    "drawdown_better_than_equal_weight",
    "statistically_significant_active_return",
    "statistically_significant_positive_alpha"
]

print("Final strategy evidence table:")
display(
    final_evidence_df[
        evidence_display_columns
    ].round(4)
)

In [ ]:
final_plot_order = [
    "CRSP S&P 500 Value Weighted",

    (
        "Quality Robustness Winner "
        "(Top 10% Inverse Volatility)"
    ),

    "Quality Baseline (Top 20% Equal Weight)",

    "CRSP S&P 500 Equal Weighted",

    "Ridge ML (Top 20% Equal Weight)",

    (
        "Six-Factor Baseline "
        "(Top 20% Equal Weight)"
    ),

    "XGBoost ML (Top 20% Equal Weight)"
]

final_color_mapping = {
    "CRSP S&P 500 Value Weighted":
        "#2ca02c",

    (
        "Quality Robustness Winner "
        "(Top 10% Inverse Volatility)"
    ):
        "#17becf",

    "Quality Baseline (Top 20% Equal Weight)":
        "#1f77b4",

    "CRSP S&P 500 Equal Weighted":
        "#7f7f7f",

    "Ridge ML (Top 20% Equal Weight)":
        "#9467bd",

    (
        "Six-Factor Baseline "
        "(Top 20% Equal Weight)"
    ):
        "#d62728",

    "XGBoost ML (Top 20% Equal Weight)":
        "#ff7f0e"
}

final_cumulative_wealth_df = (
    1.0
    + final_return_wide_df[
        final_plot_order
    ]
).cumprod()

fig, axis = plt.subplots(
    figsize=(13, 8)
)

plot_dates = (
    final_cumulative_wealth_df
    .index
    .to_numpy()
)

for series_name in final_plot_order:
    line_style = (
        "--"
        if "CRSP" in series_name
        else "-"
    )

    axis.plot(
        plot_dates,
        final_cumulative_wealth_df[
            series_name
        ].to_numpy(dtype=float),
        label=series_name,
        color=final_color_mapping[
            series_name
        ],
        linewidth=2.1,
        linestyle=line_style
    )

axis.set_title(
    "Final Out-of-Sample Strategy Comparison",
    fontsize=16,
    pad=15
)

axis.set_xlabel("Date")
axis.set_ylabel("Growth of $1 Investment")

axis.legend(
    frameon=False,
    loc="upper left",
    fontsize=9
)

axis.grid(alpha=0.25)

fig.tight_layout()

FINAL_CUMULATIVE_FIGURE = (
    FINAL_FIGURE_DIR
    / "01_final_cumulative_wealth.png"
)

fig.savefig(
    FINAL_CUMULATIVE_FIGURE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Final cumulative-wealth figure was saved.")

In [ ]:
fig, axis = plt.subplots(
    figsize=(11, 8)
)

for _, row in (
    final_performance_summary_df
    .iterrows()
):
    series_name = row["series"]

    axis.scatter(
        row["annualized_volatility"],
        row["annualized_return"],
        color=final_color_mapping[
            series_name
        ],
        s=110,
        alpha=0.9
    )

    axis.annotate(
        series_name,
        (
            row["annualized_volatility"],
            row["annualized_return"]
        ),
        xytext=(6, 5),
        textcoords="offset points",
        fontsize=8
    )

axis.set_title(
    "Annualized Risk and Return",
    fontsize=15,
    pad=15
)

axis.set_xlabel(
    "Annualized Volatility"
)

axis.set_ylabel(
    "Annualized Return"
)

axis.xaxis.set_major_formatter(
    PercentFormatter(1.0)
)

axis.yaxis.set_major_formatter(
    PercentFormatter(1.0)
)

axis.grid(alpha=0.25)

fig.tight_layout()

RISK_RETURN_FIGURE = (
    FINAL_FIGURE_DIR
    / "02_final_risk_return_comparison.png"
)

fig.savefig(
    RISK_RETURN_FIGURE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Final risk-return figure was saved.")

In [ ]:
heatmap_data_df = (
    subperiod_sharpe_pivot_df
    .reindex(final_plot_order)
)

heatmap_values = (
    heatmap_data_df
    .to_numpy(dtype=float)
)

fig, axis = plt.subplots(
    figsize=(11, 7)
)

image = axis.imshow(
    heatmap_values,
    aspect="auto",
    cmap="RdYlGn"
)

axis.set_xticks(
    np.arange(
        len(
            heatmap_data_df.columns
        )
    )
)

axis.set_xticklabels(
    heatmap_data_df.columns,
    rotation=15,
    ha="right"
)

axis.set_yticks(
    np.arange(
        len(
            heatmap_data_df.index
        )
    )
)

axis.set_yticklabels(
    heatmap_data_df.index
)

for row_index in range(
    heatmap_values.shape[0]
):
    for column_index in range(
        heatmap_values.shape[1]
    ):
        cell_value = heatmap_values[
            row_index,
            column_index
        ]

        axis.text(
            column_index,
            row_index,
            f"{cell_value:.2f}",
            ha="center",
            va="center",
            color="black",
            fontsize=9
        )

axis.set_title(
    "Subperiod Annualized Sharpe Ratios",
    fontsize=15,
    pad=15
)

color_bar = fig.colorbar(
    image,
    ax=axis
)

color_bar.set_label(
    "Annualized Sharpe Ratio"
)

fig.tight_layout()

SUBPERIOD_HEATMAP_FIGURE = (
    FINAL_FIGURE_DIR
    / "03_subperiod_sharpe_heatmap.png"
)

fig.savefig(
    SUBPERIOD_HEATMAP_FIGURE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Subperiod Sharpe heatmap was saved.")

In [ ]:
FINAL_MONTHLY_RETURN_FILE = (
    PROCESSED_DATA_DIR
    / "47_final_model_monthly_returns.csv"
)

FINAL_PERFORMANCE_FILE = (
    PROCESSED_DATA_DIR
    / "48_final_model_performance_summary.csv"
)

FINAL_SUBPERIOD_FILE = (
    PROCESSED_DATA_DIR
    / "49_final_model_subperiod_performance.csv"
)

FINAL_ACTIVE_TEST_FILE = (
    PROCESSED_DATA_DIR
    / "50_final_active_return_tests.csv"
)

FINAL_ALPHA_FILE = (
    PROCESSED_DATA_DIR
    / "51_final_factor_alpha_tests.csv"
)

FINAL_EVIDENCE_FILE = (
    PROCESSED_DATA_DIR
    / "52_final_strategy_evidence_table.csv"
)

final_evaluation_long_df.to_csv(
    FINAL_MONTHLY_RETURN_FILE,
    index=False
)

final_performance_summary_df.to_csv(
    FINAL_PERFORMANCE_FILE,
    index=False
)

final_subperiod_performance_df.to_csv(
    FINAL_SUBPERIOD_FILE,
    index=False
)

active_return_test_df.to_csv(
    FINAL_ACTIVE_TEST_FILE,
    index=False
)

final_alpha_df.to_csv(
    FINAL_ALPHA_FILE,
    index=False
)

final_evidence_df.to_csv(
    FINAL_EVIDENCE_FILE,
    index=False
)

print("Final model-comparison files were saved successfully.")

## Final Conclusions

This project developed an end-to-end equity factor-research framework
using historical S&P 500 constituents from 2005 through 2025. The
analysis covered point-in-time accounting-data alignment, factor
construction, cross-sectional validation, portfolio formation,
transaction-cost modeling, risk attribution, robustness testing, and
walk-forward machine-learning evaluation.

The pre-specified Quality Top 20% strategy generated an annualized
return of 12.51%, compared with 10.60% for the CRSP equal-weighted S&P
500 benchmark. It also achieved a higher Sharpe ratio and a smaller
maximum drawdown. Its annualized active return was 1.73%, with an
information ratio of 0.53.

However, the active-return evidence was not statistically significant
after controlling for multiple testing. The raw Newey-West p-value was
0.083, while the Holm-adjusted p-value was 0.414. The strategy's
Fama-French five-factor plus momentum alpha was also negative and
statistically insignificant. Therefore, the results support an
economically meaningful improvement relative to equal weighting, but
not the existence of a statistically reliable independent alpha.

The inverse-volatility-weighted Quality Top 10% portfolio produced the
strongest full-sample risk-adjusted performance. Nevertheless, this
configuration was identified through ex-post sensitivity analysis and
must not be interpreted as independently validated evidence. The
pre-specified Quality Top 20% equal-weighted portfolio remains the
primary research strategy.

The traditional six-factor composite failed to improve performance.
Its predictive IC was close to zero, its turnover was substantially
higher than that of the quality strategy, and several concentrated
specifications generated significantly negative raw factor-adjusted
alphas. These findings show that adding more factors does not
automatically create a stronger investment signal.

The machine-learning extension produced similarly cautious results.
Annual expanding-window Ridge and XGBoost models failed to generate
positive out-of-sample information coefficients. Ridge reduced
portfolio volatility but did not improve returns. XGBoost produced the
highest turnover, the deepest drawdown, and the lowest risk-adjusted
performance. Its negative alpha was significant before, but not after,
Holm multiple-testing correction.

Risk analysis showed that quality screening improved downside
deviation, maximum drawdown, and recovery speed relative to equal
weighting. Historical 95% VaR forecasts passed both the Kupiec
unconditional-coverage test and the Christoffersen
conditional-coverage test. However, realized tail losses moderately
exceeded CVaR forecasts during major market shocks.

Performance was strongly regime-dependent. Quality added little value
during 2015–2019, performed particularly well during the 2020–2022
stress period, and remained competitive during 2023–2025. The
capitalization-weighted benchmark delivered the highest overall
performance, driven primarily by its strong recent-period results.

Overall, the project does not support the claim that a multi-factor or
machine-learning strategy universally outperforms the market. Instead,
it provides evidence that a transparent quality screen can improve
risk-adjusted outcomes relative to an equal-weighted investment
universe, while complex factor aggregation and machine learning may
increase turnover and estimation risk without improving realized
performance.